# Ice Cream Revenue Prediction using Linear Regression

**YBI Foundation Internship Project**

This notebook performs the complete workflow:
- Load the Ice Cream dataset
- Inspect and preprocess the data
- Explore the data with visualizations
- Train a Linear Regression model
- Evaluate model performance
- Predict revenue from a user-entered temperature

> Dataset should contain `Temperature` and `Revenue` columns. If automatic download is unavailable, upload/place `Ice Cream.csv` in the notebook's working directory.


**Made by: Subhrajyoti Nayak**

## 1. Import Libraries

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="icefire")

BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "Ice Cream.csv"
MODEL_FILE = BASE_DIR / "icecream_revenue_model.pkl"
PREDICTIONS_FILE = BASE_DIR / "icecream_test_predictions.csv"


## 2. Load Dataset
The notebook first checks for a local CSV file, then attempts to download the dataset from the YBI Foundation dataset repository.

In [ ]:
DATA_URLS = [
    "https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Ice%20Cream.csv",
    "https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Ice%20Cream%20Sales%20-%20 temperatures.csv",
]

def load_dataset():
    local_candidates = [
        DATA_FILE,
        BASE_DIR / "IceCream.csv",
        BASE_DIR / "icecream.csv",
        BASE_DIR / "Ice Cream Sales.csv",
    ]

    for candidate in local_candidates:
        if candidate.exists():
            print(f"Loading dataset from: {candidate.name}")
            return pd.read_csv(candidate)

    for url in DATA_URLS:
        try:
            print("Attempting to download dataset...")
            with urllib.request.urlopen(url, timeout=12) as response:
                content = response.read()
            DATA_FILE.write_bytes(content)
            print(f"Dataset downloaded to: {DATA_FILE.name}")
            return pd.read_csv(DATA_FILE)
        except Exception:
            continue

    raise FileNotFoundError(
        "Dataset could not be downloaded. Place the YBI Ice Cream CSV file "
        "in the notebook folder as 'Ice Cream.csv'."
    )

raw_df = load_dataset()
raw_df.head()


## 3. Standardize Column Names
Map common column aliases to the required `Temperature` and `Revenue` fields.

In [ ]:
def standardize_columns(df):
    df = df.copy()
    df.columns = [str(col).strip().lower().replace("_", " ") for col in df.columns]

    aliases = {
        "temperature": ["temperature", "temp", "temperature c", "temperature (c)"],
        "revenue": ["revenue", "sales", "ice cream revenue", "icecream revenue"],
    }

    rename = {}
    for required, possible_names in aliases.items():
        for name in possible_names:
            if name in df.columns:
                rename[name] = required.title()
                break

    df.rename(columns=rename, inplace=True)
    required_cols = ["Temperature", "Revenue"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}. Found: {list(df.columns)}")

    df = df[required_cols]
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

df = standardize_columns(raw_df)
df.head()


## 4. Data Inspection and Preprocessing
Review data types, missing values, duplicates, then remove invalid or duplicate rows.

In [ ]:
print("Original dataset shape:", df.shape)
display(df.head())
print("\nData types:")
display(df.dtypes)
print("\nMissing values:")
display(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe())


In [ ]:
cleaned_df = df.dropna(subset=["Temperature", "Revenue"]).drop_duplicates().copy()
cleaned_df = cleaned_df[(cleaned_df["Temperature"] >= -50) & (cleaned_df["Temperature"] <= 60)]
cleaned_df = cleaned_df[cleaned_df["Revenue"] >= 0]

print("Cleaned dataset shape:", cleaned_df.shape)
display(cleaned_df.head())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
print(cleaned_df.describe().round(2))
print(f"Temperature–Revenue correlation: {cleaned_df['Temperature'].corr(cleaned_df['Revenue']):.4f}")


In [ ]:
# Plot 1: Temperature vs Revenue scatter plot with trend line
plt.figure(figsize=(9, 6))
sns.scatterplot(data=cleaned_df, x="Temperature", y="Revenue", s=65, alpha=0.8)
sns.regplot(data=cleaned_df, x="Temperature", y="Revenue", scatter=False, color="crimson")
plt.title("Ice Cream Revenue vs Temperature", fontsize=15, weight="bold")
plt.xlabel("Temperature")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig(BASE_DIR / "temperature_revenue_scatter.png", dpi=300)
plt.show()


In [ ]:
# Plot 2: Revenue distribution
plt.figure(figsize=(9, 5))
sns.histplot(data=cleaned_df, x="Revenue", kde=True, bins=25)
plt.title("Distribution of Ice Cream Revenue", fontsize=15, weight="bold")
plt.xlabel("Revenue")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(BASE_DIR / "revenue_distribution.png", dpi=300)
plt.show()


In [ ]:
# Plot 3: Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cleaned_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap", fontsize=14, weight="bold")
plt.tight_layout()
plt.savefig(BASE_DIR / "correlation_heatmap.png", dpi=300)
plt.show()


## 6. Split Data and Train the Linear Regression Model
Use an 80/20 train-test split with a fixed random seed for reproducibility.

In [ ]:
X = cleaned_df[["Temperature"]]
y = cleaned_df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print("Model training completed.")
print(f"Intercept: {model.intercept_:.4f}")
print(f"Coefficient (slope): {model.coef_[0]:.4f}")


## 7. Model Evaluation
Metrics shown include Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R-squared"],
    "Value": [mae, mse, rmse, r2]
})
display(metrics.round(4))


In [ ]:
# Plot 4: Actual vs predicted revenue on the test set
plt.figure(figsize=(9, 6))
plt.scatter(X_test["Temperature"], y_test, label="Actual test revenue", alpha=0.75)
order = np.argsort(X_test["Temperature"].values)
plt.plot(
    X_test["Temperature"].values[order],
    y_pred[order],
    color="crimson",
    linewidth=2.5,
    label="Linear Regression prediction",
)
plt.title("Linear Regression: Actual vs Predicted", fontsize=15, weight="bold")
plt.xlabel("Temperature")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "model_prediction_plot.png", dpi=300)
plt.show()


## 8. Save Model and Test Predictions

In [ ]:
joblib.dump(model, MODEL_FILE)

test_results = pd.DataFrame({
    "Temperature": X_test["Temperature"].values,
    "Actual Revenue": y_test.values,
    "Predicted Revenue": y_pred,
}).sort_values("Temperature")

test_results.to_csv(PREDICTIONS_FILE, index=False)

print(f"Saved trained model to: {MODEL_FILE}")
print(f"Saved test predictions to: {PREDICTIONS_FILE}")
display(test_results.head(10))


## 9. Predict Revenue for a New Temperature
Enter a temperature value in the cell below. The prediction is an estimate; values outside the dataset's temperature range are extrapolations.

In [ ]:
# Change this value to the temperature you want to test.
user_temperature = 30.0

prediction = model.predict(pd.DataFrame({"Temperature": [user_temperature]}))[0]
print(f"Estimated ice cream revenue at temperature {user_temperature:.2f}: {prediction:.2f}")

temp_min = cleaned_df["Temperature"].min()
temp_max = cleaned_df["Temperature"].max()
if user_temperature < temp_min or user_temperature > temp_max:
    print("Warning: This temperature is outside the training data range.")


## 10. Project Conclusion

This project uses **Linear Regression** to estimate ice cream revenue from temperature. Model performance should be interpreted using the evaluation metrics above. Because the model uses only one predictor, additional variables (such as season, location, promotions, or customer traffic) could improve future predictions.

### Files generated
- `icecream_revenue_model.pkl` — trained model
- `icecream_test_predictions.csv` — test-set predictions
- `temperature_revenue_scatter.png`
- `revenue_distribution.png`
- `correlation_heatmap.png`
- `model_prediction_plot.png`
